In [ ]:
pip install openpyxl

In [ ]:
# CALCULATE STABILITY OF RESULTS STEP 1
from openai import OpenAI
import json
import re
import os

# --- CONFIGURATION ---
api_keys_path = "../config/api_keys.json"
with open(api_keys_path, "r", encoding="utf-8") as f:
    api_key = json.load(f)["api_openai"]

client = OpenAI(api_key=api_key)

requirement_id = "11"

# --- FILE PATHS TO COMPARE ---
paths = [
    "" + requirement_id + ".json",
    "" + requirement_id + ".json",
    "" + requirement_id + ".json",
    "" + requirement_id + ".json",
    "" + requirement_id + ".json"
]

# --- SYSTEM PROMPT ---
system_prompt = """
You will receive multiple JSON arrays, each labeled with a unique name (e.g., iteration1, iteration2, iteration3, etc.).
Each array contains multiple requirements, each with fields such as id, precondition, norms, and temporal_validity.

- Do not compare entries *within* a single JSON array (e.g., do not compare r14_1 vs r14_2 inside iteration1).
- Only compare entire JSON arrays *against each other* (e.g., iteration1 vs iteration2).

Please do the following:

1. Compare each JSON array against every other array (pairwise).
2. Identify **any differences** between the arrays, including:
   - Changes in any field values across arrays
   - Added or missing entries (e.g., one array has 3 items, another has 2)
   - Any content mismatch for the same `id`
3. Present the results in a table with these columns:
   - Compared JSONs (e.g., iteration1 vs iteration2)
   - Are they identical? (Yes / No)
   - Field or path where the difference occurs (e.g., `norms[0].action.activities[0]`)
   - Description of the difference
4. If some JSON arrays are fully identical, group them together.

Paste your JSON arrays one after another and label them like:

iteration1:
[ ... ]

iteration2:
[ ... ]

Here are the names of the JSONs and JSON arrays to compare:
"""

# --- FORMAT INPUT FOR GPT ---
def extract_label(path):
    exec_match = re.search(r"output_(\d\w+)_execution", path)
    req_match = re.search(r"requirements_(r\d+)\.json", path)
    exec_label = exec_match.group(1) if exec_match else "unknown_exec"
    req_label = req_match.group(1) if req_match else "unknown_req"
    return f"{exec_label}_{req_label}"

input_sections = []

for path in paths:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        label = extract_label(path)
        input_sections.append(f"{label}:\n{json.dumps(data, indent=2, ensure_ascii=False)}")
    else:
        print(f"File not found: {path}")

# Combine everything
user_input = "\n\n".join(input_sections)

# --- QUERY GPT ---
def query_llm(system_prompt, user_input, model="gpt-4.1", temperature=0.0):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_input.strip()}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

# Run it
reply = query_llm(system_prompt, user_input)
print(reply)

# Save reply to file
output_file_path = "stability_results_r" + requirement_id + ".txt"
with open(output_file_path, "w", encoding="utf-8") as f:
    f.write(reply)

print(f" LLM comparison results saved to: {output_file_path}")

In [ ]:
# CALCULATE STABILITY OF RESULTS STEP 3
from openai import OpenAI
import json
import os
import pandas as pd

# --- CONFIGURATION ---
api_keys_path = "../config/api_keys.json"
ground_truth_excel = ""

requirement_id = "r1"  

base_path = ""
execution_folders = [
    "output_1st_execution",
    "output_2nd_execution",
    "output_3rd_execution",
    "output_4th_execution",
    "output_5th_execution"
]
scenario_subpath_template = "03_compliance_deviations/emergencies_scenario/deviations_{req_id}.json"

# --- Load API key ---
with open(api_keys_path, "r", encoding="utf-8") as f:
    api_key = json.load(f)["api_openai"]
client = OpenAI(api_key=api_key)

# --- Load Ground Truth from Excel ---
df = pd.read_excel(ground_truth_excel, dtype=str)
ground_truth_row = df[df["requirement_id"] == requirement_id]
if ground_truth_row.empty:
    raise ValueError(f"No ground truth entry found for requirement_id '{requirement_id}'")
ground_truth_deviation = ground_truth_row["deviations"].values[0].strip()

# --- Load First Deviation from Each Execution ---
input_sections = []
for folder in execution_folders:
    full_path = os.path.join(base_path, folder, scenario_subpath_template.format(req_id=requirement_id))
    if os.path.exists(full_path):
        with open(full_path, "r", encoding="utf-8") as f:
            deviations = json.load(f)
            if deviations:
                first_deviation = deviations[0]
                label = f"{requirement_id} - {folder}"
                input_sections.append(f"{label}:\n{json.dumps(first_deviation, indent=2, ensure_ascii=False)}")
            else:
                print(f"No deviations found in file: {full_path}")
    else:
        print(f"File not found: {full_path}")

user_input = "\n\n".join(input_sections)

# --- System Prompt Template ---
system_prompt = f"""
You are given:

1. A **ground truth deviation statement** that reflects the correct interpretation of a deviation resulting from a change in a requirement.
2. A list of **deviation JSONs**, each describing a deviation from the same requirement change.

Your task is to evaluate how well the meaning of each deviation aligns with the meaning of the ground truth.

### Ground Truth Deviation:
"{ground_truth_deviation}"

### For Each JSON, Assess the Following Semantically:

1. **Deviation Type Match**
2. **Semantic Equivalence of the Root Cause**
3. **Interpretation of the Updated Requirement**
4. **Interpretation of the Process Behavior**
5. **Alignment of Process Model Reference**
6. **Semantic Alignment Score** (1–5)

### Output Format per JSON:

- **{requirement_id} - execution_folder**
  - Deviation Type Match: Yes / No
  - Root Cause Match: Yes / Partial / No
  - Updated Requirement Interpretation: Correct / Partially Correct / Incorrect
  - Process Behavior Interpretation: Correct / Partially Correct / Incorrect
  - Process Model Reference: Aligned / Equivalent / Not Aligned
  - Semantic Alignment Score: X/5
  - **Summary Comment**: [Explain where the deviation meaning aligns or diverges from the ground truth.]

### At the End, Generate a Summary Table:

Include a compact table with the following columns:

| Execution         | Deviation Type | Root Cause | Updated Requirement | Process Behavior | Model Reference | Score | Extra Undetected Changes? | Non-impacting Changes Present? |
|-------------------|----------------|------------|----------------------|------------------|------------------|--------|----------------------------|-------------------------------|
| output_1st_execution | Yes / No      | Yes / Partial / No | Correct / Partial / Incorrect | Correct / Partial / Incorrect | Aligned / Equivalent / Not Aligned | X/5 | Yes / No | Yes / No |
| ...               |                |            |                      |                  |                  |        |                            |                               |

Then:

1. **Briefly summarize which executions are closest and furthest from the ground truth** based on semantic alignment score.

2. **Explicitly list any executions where**:
   - Changes occurred in the process or requirement model **without any deviation being triggered**. Mark them as:
     - `Extra Undetected Changes? = Yes` if meaningful changes seem to be missed by deviation detection.
     - `Non-impacting Changes Present? = Yes` if there are changes with no semantic relevance to compliance.

3. Explain what kind of **non-impacting changes** were observed (e.g., formatting, metadata, ordering, renaming).
"""

# --- Query LLM ---
def query_llm(system_prompt, user_input, model="gpt-4.1", temperature=0.0):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_input.strip()}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

reply = query_llm(system_prompt, user_input)
print(reply)

# --- Save Output ---
output_path = f"deviation_semantic_evaluation_{requirement_id}.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(reply)

print(f"Deviation evaluation results saved to: {output_path}")
